# 🍌 Detector de Calidad de Frutas — SENATI Taller ML
### Entrenamiento CNN con MobileNetV2 + Exportación a TensorFlow.js

**Pasos del notebook:**
1. Instalación de dependencias
2. Subir dataset (ZIP con carpetas: `maduro/`, `verde/`, `podrido/`)
3. Exploración y visualización del dataset
4. Preprocesamiento y Data Augmentation
5. Modelo MobileNetV2 (Transfer Learning)
6. Entrenamiento Fase 1 — cabeza clasificadora
7. Entrenamiento Fase 2 — fine-tuning
8. Evaluación (accuracy, matriz de confusión)
9. Exportación a TensorFlow.js
10. Descarga del modelo

---
> ⚡ Asegúrate de activar la GPU: **Entorno de ejecución → Cambiar tipo de entorno → T4 GPU**

## Celda 1 — Instalación de dependencias

In [ ]:
!pip install tensorflowjs scikit-learn seaborn -q

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import os, zipfile, json, shutil
from pathlib import Path
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

print(f'TensorFlow: {tf.__version__}')
print(f'GPU disponible: {tf.config.list_physical_devices("GPU")}')

## Celda 2 — Descargar dataset desde Kaggle (sin subir el ZIP)

**Mucho más rápido** que subir el ZIP desde tu PC — Colab descarga directo de Kaggle.

**Necesitas:**
1. Ir a [kaggle.com](https://kaggle.com) → tu foto → **Settings** → sección **API** → **Create New Token**
2. Se descarga un archivo pequeño llamado `kaggle.json`
3. Subirlo cuando la celda lo pida

**Configura `FRUTAS`** con las frutas que quieras incluir:

| Valor | Carpetas en el dataset |
|-------|----------------------|
| `'banana'` | freshbanana / rottenbanana |
| `'apple'` | freshapples / rottenapples |
| `'orange'` | freshoranges / rottenoranges |

> Por defecto entrena con **plátano + manzana + naranja** (las 3 frutas del dataset).  
> Puedes quitar cualquiera eliminándola de la lista `FRUTAS`.

> ⚠️ El dataset solo tiene fruta **fresca** y **podrida**, no verde/inmadura.  
> El 35 % de las imágenes frescas de cada fruta se asignará automáticamente a la clase `verde`.

In [ ]:
from google.colab import files
import random

# ── CONFIGURACIÓN ─────────────────────────────────────────────────────────────
FRUTAS           = ['banana', 'apple', 'orange']  # frutas a incluir en el entrenamiento
PORCENTAJE_VERDE = 0.35                            # 35 % de las frescas → clase "verde"
DATASET_KAGGLE   = 'sriramr/fruits-fresh-and-rotten-for-classification'
# ──────────────────────────────────────────────────────────────────────────────

# ── Paso 1: Instalar Kaggle y configurar credenciales ─────────────────────────
!pip install kaggle -q

print('📄 Sube tu archivo kaggle.json (Kaggle → Settings → API → Create New Token)')
uploaded = files.upload()

import os, shutil, zipfile
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('✅ Credenciales configuradas')

# ── Paso 2: Descargar el dataset directamente en Colab ────────────────────────
print(f'\n⬇️  Descargando dataset desde Kaggle...')
print('   (tarda 1-3 minutos según la velocidad de los servidores de Kaggle)\n')

for d in ['dataset', 'dataset_tmp']:
    if os.path.exists(d):
        shutil.rmtree(d)

os.makedirs('dataset_tmp', exist_ok=True)
!kaggle datasets download -d {DATASET_KAGGLE} -p dataset_tmp --unzip
print('✅ Descarga y descompresión completada')

# ── Paso 3: Buscar carpetas fresh/rotten para TODAS las frutas elegidas ────────
fresh_dirs  = []
rotten_dirs = []

for root, dirs, _ in os.walk('dataset_tmp'):
    for d in dirs:
        d_lower = d.lower()
        # ¿Pertenece a alguna de las frutas seleccionadas?
        es_fruta_elegida = any(fruta.lower() in d_lower for fruta in FRUTAS)
        if not es_fruta_elegida:
            continue
        full_path = os.path.join(root, d)
        if any(k in d_lower for k in ['fresh', 'ripe', 'good']):
            fresh_dirs.append(full_path)
        elif any(k in d_lower for k in ['rotten', 'spoiled', 'bad', 'old']):
            rotten_dirs.append(full_path)

# Si no encontró nada, mostrar carpetas disponibles para orientar al usuario
if not fresh_dirs and not rotten_dirs:
    print(f'\n⚠️  No se encontraron carpetas para FRUTAS={FRUTAS}.')
    print('Carpetas disponibles en el dataset:')
    todas = set()
    for root, dirs, _ in os.walk('dataset_tmp'):
        for d in dirs:
            if not d.startswith('.') and not d.startswith('__'):
                todas.add(d)
    for nombre in sorted(todas):
        print(f'   {nombre}')
    raise ValueError('Ajusta la lista FRUTAS con los nombres que aparecen en las carpetas.')

print(f'\nFrutas seleccionadas : {FRUTAS}')
print(f'Carpetas frescas     : {[os.path.basename(d) for d in fresh_dirs]}')
print(f'Carpetas podridas    : {[os.path.basename(d) for d in rotten_dirs]}')

# ── Paso 4: Recolectar imágenes de todas las carpetas ─────────────────────────
def recolectar(dirs):
    imgs = []
    exts = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')
    for d in dirs:
        for f in os.listdir(d):
            if f.lower().endswith(exts):
                imgs.append(os.path.join(d, f))
    return imgs

fresh_imgs  = recolectar(fresh_dirs)
rotten_imgs = recolectar(rotten_dirs)

# ── Paso 5: Dividir frescas → maduro (65 %) + verde (35 %) ───────────────────
random.seed(42)
random.shuffle(fresh_imgs)

n_verde     = int(len(fresh_imgs) * PORCENTAJE_VERDE)
verde_imgs  = fresh_imgs[:n_verde]
maduro_imgs = fresh_imgs[n_verde:]

# ── Paso 6: Crear dataset final con 3 clases ──────────────────────────────────
for clase in ['maduro', 'verde', 'podrido']:
    os.makedirs(f'dataset/{clase}', exist_ok=True)

def copiar(img_list, clase):
    for i, src in enumerate(img_list):
        ext = os.path.splitext(src)[1] or '.jpg'
        shutil.copy(src, f'dataset/{clase}/{clase}_{i:05d}{ext}')

copiar(maduro_imgs, 'maduro')
copiar(verde_imgs,  'verde')
copiar(rotten_imgs, 'podrido')
shutil.rmtree('dataset_tmp')

# ── Resumen ───────────────────────────────────────────────────────────────────
print(f'\n✅ Dataset listo con {len(FRUTAS)} fruta(s): {", ".join(FRUTAS).upper()}\n')
total = 0
for clase in ['maduro', 'verde', 'podrido']:
    n = len(os.listdir(f'dataset/{clase}'))
    total += n
    barra = '█' * (n // 60)
    print(f'   {clase:10s} → {n:5d} imágenes  {barra}')
print(f'\n   {"TOTAL":10s} → {total:5d} imágenes')

print(f"""
⚠️  NOTA sobre la clase "verde":
   El dataset no incluye fruta verde/inmadura de forma separada.
   Se tomó el {PORCENTAJE_VERDE*100:.0f}% de las imágenes FRESCAS para la clase "verde".
   Esto cubre la funcionalidad del prototipo.
   Para mayor precisión: reemplaza dataset/verde/ con fotos reales de fruta sin madurar.
""")

## Celda 3 — Exploración visual del dataset

In [ ]:
clases = sorted(os.listdir('dataset'))
print(f'Clases encontradas: {clases}')

fig, axes = plt.subplots(len(clases), 5, figsize=(14, 3 * len(clases)))
fig.suptitle('Muestras del dataset', fontsize=14, fontweight='bold')

for i, clase in enumerate(clases):
    imagenes = [f for f in os.listdir(f'dataset/{clase}')
                if f.lower().endswith(('.jpg','.jpeg','.png','.webp'))][:5]
    for j, img_name in enumerate(imagenes):
        ax = axes[i][j] if len(clases) > 1 else axes[j]
        img = mpimg.imread(f'dataset/{clase}/{img_name}')
        ax.imshow(img)
        ax.axis('off')
        if j == 0:
            ax.set_title(clase.upper(), fontweight='bold', color='green' if clase=='maduro' else ('orange' if clase=='verde' else 'red'))

plt.tight_layout()
plt.show()

# Distribución de clases
conteos = {c: len(os.listdir(f'dataset/{c}')) for c in clases}
colores = ['#4ade80' if c=='maduro' else '#fbbf24' if c=='verde' else '#f87171' for c in clases]

plt.figure(figsize=(6, 4))
bars = plt.bar(conteos.keys(), conteos.values(), color=colores, edgecolor='white', linewidth=1.2)
for bar, v in zip(bars, conteos.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(v), ha='center', fontweight='bold')
plt.title('Distribución de imágenes por clase')
plt.ylabel('Cantidad de imágenes')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Celda 4 — Configuración y Generadores de Datos

In [ ]:
# ── Hiperparámetros ──────────────────────────────────────────────
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
VAL_SPLIT  = 0.2       # 80% entrenamiento, 20% validación
SEED       = 42
DATA_DIR   = 'dataset'
# ─────────────────────────────────────────────────────────────────

# Generador de ENTRENAMIENTO con augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=VAL_SPLIT,
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.75, 1.25],
    fill_mode='nearest',
)

# Generador de VALIDACIÓN (solo normalización, sin augmentation)
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=VAL_SPLIT,
)

train_ds = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=SEED,
    shuffle=True,
)

val_ds = val_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=SEED,
    shuffle=False,
)

print(f'\nImágenes de entrenamiento : {train_ds.samples}')
print(f'Imágenes de validación    : {val_ds.samples}')
print(f'\n⚠️  ORDEN DE CLASES (anótalo para Angular):')
for clase, idx in sorted(train_ds.class_indices.items(), key=lambda x: x[1]):
    print(f'   índice {idx} → "{clase}"')

# Guardar el mapeo para usarlo en Angular
class_indices = train_ds.class_indices
idx_to_class = {v: k for k, v in class_indices.items()}
print(f'\n→ En Angular, probs[0]="{idx_to_class[0]}", probs[1]="{idx_to_class[1]}", probs[2]="{idx_to_class[2]}"')

## Celda 5 — Construcción del modelo (MobileNetV2)

In [ ]:
NUM_CLASES = len(train_ds.class_indices)

# ── Base preentrenada en ImageNet ──
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False   # congelar en fase 1

# ── Cabeza clasificadora ──
inputs  = tf.keras.Input(shape=(224, 224, 3))
x       = base_model(inputs, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.BatchNormalization()(x)
x       = layers.Dropout(0.35)(x)
x       = layers.Dense(128, activation='relu')(x)
x       = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs, name='detector_frutas')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f'Parámetros totales        : {model.count_params():,}')
print(f'Parámetros entrenables    : {sum(tf.size(w).numpy() for w in model.trainable_weights):,}')
print(f'Parámetros no entrenables : {sum(tf.size(w).numpy() for w in model.non_trainable_weights):,}')

## Celda 6 — Entrenamiento Fase 1 (base congelada)

In [ ]:
callbacks_fase1 = [
    callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        'mejor_fase1.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=0
    ),
]

print('=== FASE 1: Entrenando solo la cabeza clasificadora ===')
historia1 = model.fit(
    train_ds,
    epochs=25,
    validation_data=val_ds,
    callbacks=callbacks_fase1,
)

print(f'\nMejor val_accuracy fase 1: {max(historia1.history["val_accuracy"]):.4f}')

## Celda 7 — Entrenamiento Fase 2 (fine-tuning)

In [ ]:
# Descongelar las últimas 40 capas de MobileNetV2
base_model.trainable = True
capas_a_congelar = len(base_model.layers) - 40
for layer in base_model.layers[:capas_a_congelar]:
    layer.trainable = False

entrenables = sum(1 for l in model.layers if l.trainable)
print(f'Capas entrenables tras descongelar: {entrenables}')

# Learning rate mucho más bajo para no destruir los pesos preentrenados
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_fase2 = [
    callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.4,
        patience=3,
        min_lr=1e-8,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        'mejor_fase2.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=0
    ),
]

print('\n=== FASE 2: Fine-tuning (últimas 40 capas) ===')
historia2 = model.fit(
    train_ds,
    epochs=20,
    validation_data=val_ds,
    callbacks=callbacks_fase2,
)

print(f'\nMejor val_accuracy fase 2: {max(historia2.history["val_accuracy"]):.4f}')

## Celda 8 — Curvas de entrenamiento

In [ ]:
# Unir historial de ambas fases
acc   = historia1.history['accuracy']   + historia2.history['accuracy']
val_a = historia1.history['val_accuracy'] + historia2.history['val_accuracy']
loss  = historia1.history['loss']       + historia2.history['loss']
val_l = historia1.history['val_loss']   + historia2.history['val_loss']
sep   = len(historia1.history['accuracy'])   # época donde empieza fase 2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Curvas de Entrenamiento', fontsize=13, fontweight='bold')

epochs_total = range(1, len(acc) + 1)

for ax, train_vals, val_vals, titulo, ylabel in [
    (ax1, acc,  val_a, 'Exactitud (Accuracy)', 'Accuracy'),
    (ax2, loss, val_l, 'Pérdida (Loss)',        'Loss'),
]:
    ax.plot(epochs_total, train_vals, '#4ade80', linewidth=2, label='Entrenamiento')
    ax.plot(epochs_total, val_vals,   '#60a5fa', linewidth=2, label='Validación')
    ax.axvline(sep, color='#fbbf24', linestyle='--', alpha=0.7, label=f'Fine-tuning (ép. {sep})')
    ax.set_title(titulo); ax.set_xlabel('Época'); ax.set_ylabel(ylabel)
    ax.legend(); ax.grid(alpha=0.3)
    ax.set_facecolor('#0d1117')

fig.patch.set_facecolor('#161b22')
plt.tight_layout()
plt.show()

val_loss, val_acc = model.evaluate(val_ds, verbose=0)
print(f'\n✅ Exactitud final en validación: {val_acc*100:.2f}%')
print(f'   Pérdida  final en validación: {val_loss:.4f}')

## Celda 9 — Evaluación detallada

In [ ]:
val_ds.reset()
y_pred_probs = model.predict(val_ds, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = val_ds.classes

clases_orden = [k for k, v in sorted(class_indices.items(), key=lambda x: x[1])]

print('=== REPORTE DE CLASIFICACIÓN ===')
print(classification_report(y_true, y_pred, target_names=clases_orden))

# Matriz de confusión
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=clases_orden, yticklabels=clases_orden, ax=ax1)
ax1.set_title('Matriz de Confusión (valores absolutos)')
ax1.set_ylabel('Real'); ax1.set_xlabel('Predicho')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=clases_orden, yticklabels=clases_orden, ax=ax2)
ax2.set_title('Matriz de Confusión (proporciones)')
ax2.set_ylabel('Real'); ax2.set_xlabel('Predicho')

plt.tight_layout()
plt.show()

## Celda 10 — Predicciones de ejemplo (verificación visual)

In [ ]:
from tensorflow.keras.preprocessing import image as keras_image

colores_clase = {'maduro': '#4ade80', 'verde': '#fbbf24', 'podrido': '#f87171'}

fig, axes = plt.subplots(3, 5, figsize=(15, 9))
fig.suptitle('Predicciones sobre imágenes de validación', fontsize=13, fontweight='bold')

idx = 0
for clase_real_idx, clase_real in enumerate(clases_orden):
    imagenes_clase = [p for p, c in zip(val_ds.filepaths, val_ds.classes) if c == clase_real_idx][:5]
    for j, img_path in enumerate(imagenes_clase):
        ax = axes[clase_real_idx][j]
        img = keras_image.load_img(img_path, target_size=IMG_SIZE)
        img_arr = keras_image.img_to_array(img) / 255.0
        pred = model.predict(img_arr[np.newaxis, ...], verbose=0)[0]
        pred_idx = np.argmax(pred)
        pred_clase = clases_orden[pred_idx]
        confianza = pred[pred_idx] * 100
        correcto = pred_clase == clase_real

        ax.imshow(img)
        ax.axis('off')
        color = '#4ade80' if correcto else '#f87171'
        icono = '✓' if correcto else '✗'
        ax.set_title(f'{icono} {pred_clase}\n{confianza:.0f}%',
                     fontsize=9, color=color, fontweight='bold')
        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(2)

plt.tight_layout()
plt.show()

## Celda 11 — Guardar y exportar a TensorFlow.js (graph-model)

> ⚠️ Se usa `tf.saved_model.save` + `tfjs_graph_model` en lugar de `save_keras_model`.  
> Esto evita el problema de incompatibilidad entre **Keras 3** y TF.js en el navegador.

In [ ]:
import subprocess

# ── 1. Guardar backup en formato .h5 ──────────────────────────────────────────
model.save('modelo_frutas.h5')
print('✅ Backup guardado: modelo_frutas.h5')

# ── 2. Limpiar carpetas ───────────────────────────────────────────────────────
for d in ['modelo_tfjs', 'modelo_savedmodel']:
    if os.path.exists(d):
        shutil.rmtree(d)
os.makedirs('modelo_tfjs')

# ── 3. Exportar como TF SavedModel estándar ───────────────────────────────────
print('\nExportando como TF SavedModel...')
tf.saved_model.save(model, 'modelo_savedmodel')
print('✅ TF SavedModel guardado')

# ── 4. Convertir a tfjs_graph_model ───────────────────────────────────────────
print('\nConvirtiendo a TF.js graph-model...')
r = subprocess.run(
    [
        'tensorflowjs_converter',
        '--input_format',  'tf_saved_model',
        '--output_format', 'tfjs_graph_model',
        'modelo_savedmodel',
        'modelo_tfjs',
    ],
    capture_output=True, text=True
)

if r.returncode != 0:
    print('❌ Error en la conversión:')
    print(r.stderr[-600:])
    raise RuntimeError('Conversión fallida. Revisa el error anterior.')

# ── 5. Verificar formato ──────────────────────────────────────────────────────
with open('modelo_tfjs/model.json') as f:
    fmt = json.load(f).get('format', '?')

if fmt != 'graph-model':
    raise RuntimeError(f'Formato incorrecto: {fmt!r}. Se esperaba "graph-model".')

print(f'✅ Conversión exitosa — formato: {fmt}')
print('\n📦 Archivos generados:')
total_kb = 0
for fname in sorted(os.listdir('modelo_tfjs')):
    size_kb = os.path.getsize(f'modelo_tfjs/{fname}') / 1024
    total_kb += size_kb
    print(f'   {fname:<45s} {size_kb:7.1f} KB')
print(f'   {"TOTAL":<45s} {total_kb:7.1f} KB')

## Celda 12 — Guardar metadatos del modelo

In [ ]:
# Guardar metadatos útiles para Angular
val_loss_final, val_acc_final = model.evaluate(val_ds, verbose=0)

metadata = {
    'clases': clases_orden,
    'class_indices': class_indices,
    'idx_to_class': {str(v): k for k, v in class_indices.items()},
    'input_size': [224, 224, 3],
    'val_accuracy': round(float(val_acc_final), 4),
    'val_loss': round(float(val_loss_final), 4),
    'modelo_base': 'MobileNetV2',
    'num_clases': NUM_CLASES,
    'frutas': FRUTAS,
    'descripcion': f'Detector de calidad de frutas — maduro/verde/podrido ({", ".join(FRUTAS)})',
}

with open('modelo_tfjs/metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print('✅ Metadatos guardados en metadata.json')
print(json.dumps(metadata, indent=2, ensure_ascii=False))

## Celda 13 — Verificación rápida del modelo TF.js

In [ ]:
from tensorflow.keras.preprocessing import image as keras_image

# ── Verificar model.json ──────────────────────────────────────────────────────
with open('modelo_tfjs/model.json') as f:
    model_json = json.load(f)

formato   = model_json.get('format', 'N/A')
generado  = model_json.get('generatedBy', 'N/A')
convertido = model_json.get('convertedBy', 'N/A')

print('✅ model.json es válido')
print(f'   Formato     : {formato}')
print(f'   Generado por: {generado}')
print(f'   Convertido  : {convertido}')

if formato == 'graph-model':
    print('\n🟢 Formato correcto — TF.js lo cargará sin problemas en el navegador')
else:
    print(f'\n🔴 ADVERTENCIA: el formato es "{formato}", no "graph-model"')
    print('   Vuelve a ejecutar la Celda 11')

# ── Prueba de inferencia en Python (simulada) ─────────────────────────────────
print('\n🧪 Prueba de inferencia con una imagen de validación:')
sample_path  = val_ds.filepaths[0]
sample_clase = clases_orden[val_ds.classes[0]]

img     = keras_image.load_img(sample_path, target_size=IMG_SIZE)
img_arr = keras_image.img_to_array(img) / 255.0
probs   = model.predict(img_arr[np.newaxis, ...], verbose=0)[0]

print(f'   Imagen real : {sample_clase}')
for clase, prob in zip(clases_orden, probs):
    barra = '█' * int(prob * 30)
    print(f'   {clase:10s} : {prob*100:5.1f}%  {barra}')
print(f'\n   Predicción  : {clases_orden[np.argmax(probs)]} ({np.max(probs)*100:.1f}% confianza)')

## Celda 14 — Descargar el modelo

Descarga el ZIP con todos los archivos necesarios para Angular.

In [ ]:
# Crear ZIP del modelo TF.js
shutil.make_archive('modelo_para_angular', 'zip', 'modelo_tfjs')

size_mb = os.path.getsize('modelo_para_angular.zip') / (1024 * 1024)
print(f'📦 modelo_para_angular.zip ({size_mb:.1f} MB) listo para descargar')
print()
print('📋 INSTRUCCIONES:')
print('   1. Descarga el archivo con la línea siguiente')
print('   2. Descomprime el ZIP')
print('   3. Copia los archivos a: detector-frutas/src/assets/model/')
print('   4. La carpeta debe quedar así:')
print('      src/assets/model/')
print('        model.json')
print('        group1-shard1of1.bin  (puede haber más .bin)')
print('        metadata.json')

files.download('modelo_para_angular.zip')

## Celda 15 — Código para Angular (integración real)

Este es el código que debes usar en `model.ts` para reemplazar el mock y usar el modelo real.

In [ ]:
# Leer el orden de clases del metadata
with open('modelo_tfjs/metadata.json') as f:
    meta = json.load(f)

idx_to_class = meta['idx_to_class']

code = f'''
// ──────────────────────────────────────────────
// Integración real en ModelService (model.ts)
// ──────────────────────────────────────────────
// Orden de clases del modelo entrenado:
//   0 → "{idx_to_class['0']}"
//   1 → "{idx_to_class['1']}"
//   2 → "{idx_to_class['2']}"

import * as tf from '@tensorflow/tfjs';

// En la clase ModelService:
private tfModel: tf.LayersModel | null = null;

async loadModel(): Promise<void> {{
  this.tfModel = await tf.loadLayersModel('assets/model/model.json');
  this.modelLoaded.set(true);
  console.log('Modelo cargado ✓');
}}

// Reemplaza mockPredict() por este método:
private async realPredict(
  imgEl: HTMLImageElement | HTMLVideoElement | HTMLCanvasElement
): Promise<number[]> {{
  if (!this.tfModel) throw new Error('Modelo no cargado');

  const tensor = tf.tidy(() =>
    tf.browser
      .fromPixels(imgEl)
      .resizeNearestNeighbor([224, 224])
      .toFloat()
      .div(255)
      .expandDims(0)
  );

  const output = this.tfModel.predict(tensor) as tf.Tensor;
  const probs  = Array.from(await output.data());
  tensor.dispose();
  output.dispose();

  // probs[0] = "{idx_to_class['0']}"
  // probs[1] = "{idx_to_class['1']}"
  // probs[2] = "{idx_to_class['2']}"
  return probs;
}}
'''

print(code)

---
## ✅ Resumen final

| Paso | Estado |
|------|--------|
| Dataset cargado | ✓ |
| Modelo MobileNetV2 construido | ✓ |
| Entrenamiento Fase 1 | ✓ |
| Fine-tuning Fase 2 | ✓ |
| Evaluación (matriz de confusión) | ✓ |
| Exportación a TF.js | ✓ |
| Descarga del modelo | ✓ |

### Próximos pasos en tu proyecto Angular
1. Descomprime `modelo_para_angular.zip`
2. Copia los archivos a `src/assets/model/`
3. En `model.ts`, llama a `loadModel()` en el constructor y usa `realPredict()` en `runInference()`
4. Ajusta el mapeo de índices según lo que mostró la **Celda 4** (`ORDEN DE CLASES`)

### Si el accuracy es bajo (<75%)
- Agrega más imágenes (objetivo: 500+ por clase)
- Asegúrate de que las fotos tienen variedad de ángulos, iluminación y fondo
- Aumenta el data augmentation en la Celda 4
- Prueba descongelar más capas en el fine-tuning (cambia 40 → 60)